# Libraries

Just like before, lets import the libraries we'll be using. In addition to our usual libraries, we'll also import the python file we used to save our data processing functions from our source code directory (src). We can import it like a module since we've added an __init__.py file within that folder alongside adding it to our system's path list.

In [1]:
import pandas as pd
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

sys.path.append(os.path.abspath(os.pardir))

from src.data_processing import *

# Dataset Processing

Here, we'll take a look at the processed dataset. Fortunately, we've already done the data processing in our data_processing python notebook, all we need to do now is to call the process_data() function we've prepared inside our data_processing python file.

In [2]:
dataset_path = os.path.join('..','datasets','Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)
train, test = process_data(
    df = df, 
    feature_col = 'Content',
    label_col = 'Label',
    random_state = 42,
    df_name = 'Philippine Fake News Corpus.csv'
)

display(train)
display(test)

,Content,Label
0,"[397, 398, 7972, 5154, 8, 2589, 3706, 5252, 10...",1
1,"[542, 7972, 1242, 2722, 4, 1841, 2654, 6506, 7...",0
2,"[542, 7972, 196, 7075, 29, 1582, 7953, 7926, 5...",0
3,"[397, 398, 7972, 220, 4354, 137, 7964, 7934, 1...",1
4,"[397, 398, 7972, 220, 4126, 73, 6316, 790, 795...",1
...,...,...
23678,"[311, 967, 7972, 220, 2905, 2615, 27, 8, 295, ...",0
23679,"[311, 846, 7972, 220, 3219, 2805, 1639, 7926, ...",0
23680,"[542, 7972, 220, 1133, 65, 3940, 137, 5609, 79...",0
23681,"[1716, 1715, 7972, 1055, 1063, 5485, 209, 65, ...",1


,Content,Label
0,"[4020, 4110, 295, 7972, 2129, 1433, 1346, 7941...",1
1,"[1716, 1715, 7972, 716, 150, 1523, 73, 64, 566...",1
2,"[397, 398, 7972, 7419, 358, 76, 426, 6153, 65,...",1
3,"[542, 7972, 220, 1475, 133, 8, 774, 27, 1148, ...",0
4,"[397, 398, 7972, 7233, 926, 7941, 220, 397, 39...",1
...,...,...
5916,"[4020, 4110, 295, 7972, 220, 2800, 201, 853, 3...",1
5917,"[542, 7972, 57, 2330, 205, 2467, 63, 1366, 450...",0
5918,"[397, 398, 7972, 7419, 347, 1290, 466, 5836, 7...",1
5919,"[1716, 1715, 7972, 57, 420, 190, 4242, 5202, 7...",1


Our function looks functional, separating our dataset into train and test in addition to encoding them into numeric representations that we can work with. However, before we can pass this to the model, it needs to be arranged into tensors.

In addition to this, we want to pass it by batches, as such, we'll have to configure a dataloader for our model.

# Dataloader

Our dataloader will convert our dataset into batches, a sample of the actual dataset that our model can learn with in increments. Before this, we'll have to convert it to tensors, the shapes of the tensors must be consistent, which means that we'll have to pad all the sequences to have the same length.

## Converting to Tensor and Padding

Before creating a custom dataset, we need to make our values into tensors with consistent shape. Making a consistent shape can be done through padding, luckily for us, pytorch already has a dedicated function for this. All we have to do now is to convert the values into tensors.

In [3]:
train.Content = train.Content.apply(lambda x: torch.tensor(x))
train.Label = train.Label.apply(lambda x: torch.tensor(x))

test.Content = test.Content.apply(lambda x: torch.tensor(x))
test.Label = test.Label.apply(lambda x: torch.tensor(x))

display(train.head())
display(test.head())

,Content,Label
0,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
1,"[tensor(542), tensor(7972), tensor(1242), tens...",tensor(0)
2,"[tensor(542), tensor(7972), tensor(196), tenso...",tensor(0)
3,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
4,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)


,Content,Label
0,"[tensor(4020), tensor(4110), tensor(295), tens...",tensor(1)
1,"[tensor(1716), tensor(1715), tensor(7972), ten...",tensor(1)
2,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
3,"[tensor(542), tensor(7972), tensor(220), tenso...",tensor(0)
4,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)


After converting to tensors, we want to pad them. We can make use of the pad_sequence() function from pytorch to pad them to the max length in the dataset; however, if we do this separately for each dataset, we'll end up with two different lengths: one max for train and another for test. To resolve this, we'll have to briefly combine train and test, then pad them, afterwards, separate them when calling the custom Dataset class.

In [4]:
merged = pd.concat([
    train.Content,
    test.Content
])

padded = pad_sequence(merged, batch_first = True, padding_value = 3)
padded.shape

torch.Size([29604, 24186])

## Creating a Custom Dataset

Before we can make use of a dataloader, we need to create a custom dataset class that the dataloader can work with. Creating it is quite simple, as we simply need to create a pseudo-custom dataset class that has the basic dunder methods like indexing and len.

In [14]:
class FakeNewsDataset(Dataset):
    def __init__(self, feature, label):
        super().__init__()
        self.feature = feature
        self.label = label
        
    def __len__(self):
        return len(self.feature)
    
    def __getitem__(self, idx):
        feature = self.feature[idx]
        label = self.label[idx]
        
        return feature, label

In [19]:
fake_news_train = FakeNewsDataset(
    padded[:len(train),:],
    train.Label
)

fake_news_test = FakeNewsDataset(
    padded[len(train):, :],
    test.Label
)

In [20]:
fake_news_train[0], fake_news_train[0][0].shape

((tensor([ 397,  398, 7972,  ...,    3,    3,    3]), tensor(1)),
 torch.Size([24186]))

In [21]:
fake_news_test[0], fake_news_test[0][0].shape

((tensor([4020, 4110,  295,  ...,    3,    3,    3]), tensor(1)),
 torch.Size([24186]))

## Creating the Dataloader

In [22]:
train_loader = DataLoader(
    dataset = fake_news_train,
    batch_size = 32,
    shuffle = True
)

test_loader = DataLoader(
    dataset = fake_news_test,
    batch_size = 32,
    shuffle = True
)

Now that we've dealt with our dataset, all that's left is to design our model architecture. Before anything, we'll start with an embedding layer so that we can convert the indices into context vectors that the model can use to learn meaning. We'll use a similar architecture to a CNN, using 1 dimensional convolutional layers paired with max pooling layers. Afterwards, it will be passed to a flatten layer and finally to a linear layer for the final prediction.

# Model Architecture

In [ ]:
class FakeNewsDetector(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_id, conv_dim, kernel_size):
        self.embed = nn.Embedding(vocab_size, embed_dim, pad_id)
        self.conv1d = nn.Conv1d(embed_dim, conv_dim, kernel_size)
        self.pool1d = nn.MaxPool1d(kernel_size)
        self.flat = nn.Flatten()
        
        # Calculate Flattened shape
        with torch.no_grad():
            ...
        
        self.fc = nn.Linear()
        
    def forward(self, x):
        ...
        return x

# Tests

In [14]:
sample_params = {
    'batch_size': 32,
    'sequence_length': 100,
    'vocab_size': 8000,
    'embed_dim': 5,
    'pad_id': 3,
    'conv_dim': 2,
    'kernel_size': 3
}

sample_layers = {
    'embed': nn.Embedding(
        sample_params['vocab_size'],
        sample_params['embed_dim'],
        sample_params['pad_id']
    ),
    
    'conv1d': nn.Conv1d(
        sample_params['embed_dim'],
        sample_params['conv_dim'],
        sample_params['kernel_size']
    ),
    
    'pool1d': nn.MaxPool1d(sample_params['kernel_size']),
    'flat': nn.Flatten()
}

In [15]:
sample_input = torch.randint(
    low = 0, 
    high = sample_params['vocab_size'], 
    size = (sample_params['batch_size'], sample_params['sequence_length'])
)

sample_input, sample_input.shape

(tensor([[7587, 2361, 5041,  ..., 7327, 1171,  906],
         [2640, 5808, 1678,  ..., 4613, 2595, 1195],
         [ 871, 6854, 2599,  ..., 5779, 3575, 5638],
         ...,
         [6621, 3063, 5721,  ..., 5414,  930, 4111],
         [3662, 3450, 2083,  ..., 5446, 1280, 1605],
         [1642,  570, 3914,  ...,  320, 4056, 4034]]),
 torch.Size([32, 100]))

In [16]:
embed_pass = sample_layers['embed'](sample_input)
embed_pass, embed_pass.shape

(tensor([[[ 4.1230e-01,  2.9537e+00, -9.1654e-02, -1.0708e+00,  1.0465e+00],
          [ 1.3905e+00, -9.1709e-01,  6.4826e-01,  9.7102e-01,  1.5490e+00],
          [-3.0916e-01, -4.8432e-01, -8.4230e-01, -6.2037e-03,  8.4900e-01],
          ...,
          [ 5.0850e-01, -1.8057e+00,  1.5163e+00,  1.2569e+00,  1.5949e-01],
          [-5.1049e-01, -2.3837e-01, -7.3999e-01,  2.7490e-01,  1.1601e+00],
          [-2.9829e-02, -2.1448e+00,  8.5370e-01,  3.2076e-01, -9.9897e-01]],
 
         [[-7.5310e-01, -7.9495e-01,  1.0799e+00,  2.8031e-01, -2.0693e+00],
          [-4.4359e-01,  1.2421e+00, -6.5724e-01,  7.3612e-02,  1.3329e+00],
          [-4.0631e-02, -1.2011e+00, -1.5641e+00, -1.0614e+00,  1.9324e-01],
          ...,
          [-2.5817e-01,  2.8370e-01,  6.2119e-01,  5.5854e-01,  1.7084e-01],
          [ 9.5011e-01, -7.0183e-01, -5.2800e-01, -3.4326e-01,  6.0232e-01],
          [-1.7351e+00,  1.7564e+00, -9.1456e-01,  1.1895e+00,  6.5912e-01]],
 
         [[-1.3037e+00, -2.3070e-01,  4.

In [17]:
# Conv1d accepts (batch_size, channel, length)
# however, our current shape is (batch_size, length, embed_dim)
# We need to swap length with embed_dim to match conv1d

# swap dimensions 1 and 2
embed_pass_t = embed_pass.transpose(1, 2) 

conv1d_pass = sample_layers['conv1d'](embed_pass_t)
conv1d_pass, conv1d_pass.shape

(tensor([[[ 0.1374, -0.4069,  0.6837,  ..., -0.0328, -0.3758, -0.5262],
          [ 0.6166, -0.6746, -0.8250,  ...,  0.3141,  0.1438, -0.1323]],
 
         [[ 0.1222, -0.3177, -0.3945,  ..., -0.4211,  0.2279, -0.2314],
          [-0.2051, -0.0445,  0.0550,  ..., -0.0174,  0.0288, -0.3706]],
 
         [[ 0.1181,  0.2876,  0.7500,  ...,  0.5909, -0.2142,  0.9337],
          [-0.3687, -0.6167, -1.2705,  ..., -0.3561,  0.7518, -0.1911]],
 
         ...,
 
         [[ 0.7965,  0.2561, -0.4208,  ..., -0.0689,  0.7660, -0.1048],
          [ 0.2485, -0.2479, -0.6288,  ...,  0.8422, -0.4930,  0.5409]],
 
         [[ 0.6005, -0.0325, -0.5323,  ...,  0.4763,  0.5701, -0.3368],
          [-0.2101,  0.0909, -0.7106,  ..., -0.2755, -0.1666, -1.2586]],
 
         [[ 1.3027,  0.4617,  0.3074,  ...,  0.4195, -0.2560,  0.3747],
          [-0.3830, -1.0413,  0.1402,  ..., -1.2923,  0.1100, -0.3596]]],
        grad_fn=<ConvolutionBackward0>),
 torch.Size([32, 2, 98]))

In [18]:
pool1d_pass = sample_layers['pool1d'](conv1d_pass)
pool1d_pass, pool1d_pass.shape

(tensor([[[ 0.6837,  0.4732,  1.0278,  ...,  0.4093,  0.8119,  0.4110],
          [ 0.6166, -0.0922,  0.8497,  ...,  0.1626,  0.7168,  0.3141]],
 
         [[ 0.1222,  0.5632,  0.4605,  ...,  0.0570,  0.9825,  1.0944],
          [ 0.0550,  0.6714,  0.4860,  ...,  0.2994, -0.1324,  0.3237]],
 
         [[ 0.7500,  0.6123,  0.1585,  ...,  0.4433,  0.3726,  0.8063],
          [-0.3687,  0.2480,  0.2684,  ...,  0.4785,  0.1249, -0.1242]],
 
         ...,
 
         [[ 0.7965,  0.5190,  0.5715,  ...,  1.0884,  0.6551,  0.6954],
          [ 0.2485, -0.1410,  0.0245,  ...,  0.1304,  0.4784,  0.8422]],
 
         [[ 0.6005,  0.7474, -0.0681,  ...,  1.0615,  0.5910,  0.4763],
          [ 0.0909,  0.1890,  0.1017,  ...,  0.0803,  0.5309,  0.2279]],
 
         [[ 1.3027,  1.2215,  0.8491,  ...,  0.2379,  0.8659,  0.7653],
          [ 0.1402,  0.3524,  0.1056,  ..., -0.4308,  0.5921,  0.1159]]],
        grad_fn=<SqueezeBackward1>),
 torch.Size([32, 2, 32]))

In [19]:
flat_pass = sample_layers['flat'](pool1d_pass)
flat_pass, flat_pass.shape

(tensor([[ 0.6837,  0.4732,  1.0278,  ...,  0.1626,  0.7168,  0.3141],
         [ 0.1222,  0.5632,  0.4605,  ...,  0.2994, -0.1324,  0.3237],
         [ 0.7500,  0.6123,  0.1585,  ...,  0.4785,  0.1249, -0.1242],
         ...,
         [ 0.7965,  0.5190,  0.5715,  ...,  0.1304,  0.4784,  0.8422],
         [ 0.6005,  0.7474, -0.0681,  ...,  0.0803,  0.5309,  0.2279],
         [ 1.3027,  1.2215,  0.8491,  ..., -0.4308,  0.5921,  0.1159]],
        grad_fn=<ViewBackward0>),
 torch.Size([32, 64]))